![motorcycle](motorcycle.jpg)

# Motorcycle Parts Sales Analysis
### Wholesale revenue, payment economics, product mix and warehouse operations (June – August 2021)

| | |
|---|---|
| **Prepared for** | Board of Directors; Finance, Marketing and Operations leads |
| **Data source** | `sales` table: three warehouses (North, Central, West), retail and wholesale orders |
| **Period covered** | June – August 2021 |
| **Currency** | USD |

**Contents**

0. Executive summary
1. Business context and data dictionary
2. Methodology, definitions and assumptions
3. Data validation
4. Wholesale net revenue by product line, month and warehouse
5. Payment-method profitability by warehouse and month
6. Most-ordered product lines by client type
7. Product-line Pareto (ABC) analysis
8. Warehouse throughput and network load
9. Order profiling: retail vs wholesale
10. Investigation: the West warehouse Engine gap
11. Recommendations and next steps
12. Limitations

## 0. Executive summary

**Headline:** activity is concentrated in a few product lines, in the Central warehouse, and in a small number of large wholesale orders. Two items need follow-up before decisions are made: how `payment_fee` is treated in net revenue, and the absence of wholesale Engine sales at the West warehouse.

**Key findings** (volume and order figures are exact; revenue figures come from the previous execution and should refresh when the notebook is re-run)

1. **Network scale.** 1,000 orders and 9,395 units. Central handles 48.2% of units (4,527), North 34.6% (3,254) and West 17.2% (1,614). Total net revenue is about USD 289.1K.
2. **Wholesale drives volume.** Wholesale is 22.5% of orders but 55.1% of units (5,180 of 9,395). A wholesale order averages 21–24 units and USD 583–USD 754, versus 5.3–5.6 units and USD 160–USD 172 for retail.
3. **Revenue is concentrated.** Suspension & traction (25.3%) and Frame & body (23.9%) generate about 49% of net revenue; adding Electrical system takes the share to about 64%.
4. **Units and revenue tell different stories.** Braking system is the second most ordered line (2,130 units; most ordered for wholesale) but yields only 13.3% of net revenue at roughly USD 18 per unit. Engine sells just 627 units yet earns 13.1% of revenue at roughly USD 60.5 per unit, the highest in the range.
5. **Payment methods.** Transfer is the top payment method by net revenue in all nine warehouse-month combinations. This partly reflects its share of volume; the cost-efficiency view in Section 5 separates the two effects.
6. **West warehouse.** West has the smallest share of activity, its Transfer net revenue fell about 25% from June to August while Central and North each rose roughly 32–34% in August, it has no wholesale Engine sales in any month, and its wholesale average unit price (USD 27.49) is about 10–12% below Central and North.
7. **Central warehouse.** Central carries the heaviest load and its August volume was the highest of the period, making it the first candidate for congestion risk. The data has no capacity measures, so Section 8 uses peak-day load as a proxy.

**Items requiring confirmation**

- The data dictionary describes `payment_fee` as a *percentage* of `total`, while the net revenue calculation subtracts it as an amount. Run the Section 3 diagnostic and see Section 2 before circulating revenue figures.
- The Section 10 queries decide whether the West Engine gap is a supply issue or a demand issue.

## 1. Business context and data dictionary

You're working for a company that sells motorcycle parts, and they've asked for help in analyzing their sales data.

They operate three warehouses in the area, selling both retail and wholesale. They offer a variety of parts and accept credit cards, cash, and bank transfer as payment methods. Each payment type incurs a different fee.

The board of directors wants a better understanding of wholesale revenue by product line, and how it varies month to month and across warehouses. Finance wants to know which payment methods are most profitable, and Marketing wants to know which product lines to feature for each client type.

**Table: `sales`**

| Column | Data type | Description |
|--------|-----------|-------------|
| `order_number` | `VARCHAR` | Unique order number. |
| `date` | `DATE` | Date of the order, from June to August 2021. |
| `warehouse` | `VARCHAR` | The warehouse that the order was made from: `North`, `Central`, or `West`. |
| `client_type` | `VARCHAR` | Whether the order was `Retail` or `Wholesale`. |
| `product_line` | `VARCHAR` | Type of product ordered. |
| `quantity` | `INT` | Number of products ordered. |
| `unit_price` | `FLOAT` | Price per product (dollars). |
| `total` | `FLOAT` | Total price of the order (dollars). |
| `payment` | `VARCHAR` | Payment method: `Credit card`, `Transfer`, or `Cash`. |
| `payment_fee` | `FLOAT` | Percentage of `total` charged as a result of the `payment` method. |

## 2. Methodology, definitions and assumptions

| Metric | Definition |
|---|---|
| **Net revenue** | `SUM(total - payment_fee)`: order value after payment-processing fees |
| **Units** | `SUM(quantity)` |
| **Orders** | `COUNT(DISTINCT order_number)` |
| **Average units per order** | `AVG(quantity)`; each `order_number` is unique, so one row is one order for one product line |
| **Net revenue per unit** | Net revenue divided by units; separates high-value from high-volume lines |
| **Cumulative revenue share** | Running share of net revenue after sorting product lines from highest to lowest |
| **ABC class** | A: cumulative share up to 70%; B: up to 90%; C: above 90% |
| **Volume share** | A warehouse's units as a share of all units |

**Conventions**

- Months are sorted chronologically (June, July, August), not alphabetically.
- Rankings use `RANK()`, so tied values are all kept (this matters for the Retail top-3 list, where two lines tie at 862 units).
- Money is rounded to two decimals for presentation only.

**Assumption to confirm: treatment of `payment_fee`.**
The data dictionary describes `payment_fee` as a percentage of `total`. This notebook keeps the original calculation, which subtracts `payment_fee` from `total` as an amount. If the diagnostic in Section 3 shows that `payment_fee` holds small rates (for example 0.02 for 2%), every net revenue query should instead use:

```sql
SUM(total * (1 - payment_fee))   -- if the fee is stored as a fraction
SUM(total * (1 - payment_fee / 100.0))   -- if the fee is stored as a whole-number percent
```

Rankings are unlikely to change materially, but absolute net revenue and fee figures would.

## 3. Data validation

Before analysis, confirm basic data quality: row and order counts, date range, missing values, and whether `total` reconciles to `quantity × unit_price`. A second query summarizes `payment_fee` by payment method to settle the fee question described in Section 2.

In [27]:
-- Data profile: size, coverage, completeness and internal consistency
SELECT
    COUNT(*)                                                       AS row_count,
    COUNT(DISTINCT order_number)                                   AS distinct_orders,
    MIN(date)                                                      AS first_order_date,
    MAX(date)                                                      AS last_order_date,
    COUNT(*) FILTER (WHERE total IS NULL
                        OR quantity IS NULL
                        OR payment_fee IS NULL)                    AS rows_with_missing_values,
    COUNT(*) FILTER (WHERE ABS(total - quantity * unit_price) > 0.01) AS rows_where_total_differs,
    ROUND(SUM(total)::numeric, 2)                                  AS gross_revenue
FROM sales;

,row_count,distinct_orders,first_order_date,last_order_date,rows_with_missing_values,rows_where_total_differs,gross_revenue
0,1000,1000,2021-06-01 00:00:00+00:00,2021-08-28 00:00:00+00:00,0,431,289113.0


In [28]:
-- Payment fee diagnostic: what does payment_fee actually contain
SELECT
    payment,
    COUNT(DISTINCT order_number)                                         AS orders,
    ROUND(SUM(total)::numeric, 2)                                        AS gross_revenue,
    ROUND(SUM(payment_fee)::numeric, 2)                                  AS total_fees,
    ROUND(SUM(total - payment_fee)::numeric, 2)                          AS net_revenue,
    MIN(payment_fee)                                                     AS min_fee_value,
    MAX(payment_fee)                                                     AS max_fee_value,
    ROUND((100 * SUM(payment_fee) / NULLIF(SUM(total), 0))::numeric, 2)  AS fees_pct_of_gross,
    ROUND((100.0 * SUM(total) / NULLIF(SUM(SUM(total)) OVER (), 0))::numeric, 1) AS share_of_gross_pct
FROM sales
GROUP BY payment
ORDER BY net_revenue DESC;

,payment,orders,gross_revenue,total_fees,net_revenue,min_fee_value,max_fee_value,fees_pct_of_gross,share_of_gross_pct
0,Transfer,225,159642.33,2.25,159640.08,0.01,0.01,0.00,55.2
1,Credit card,659,110271.57,19.77,110251.80,0.03,0.03,0.02,38.1
2,Cash,116,19199.10,0.00,19199.10,0.00,0.00,0.00,6.6


**How to read the diagnostic.** Card and transfer fees are typically in the low single-digit percent range. If `fees_pct_of_gross` is close to zero (well below 1%) while `min_fee_value` and `max_fee_value` are small decimals, the column holds rates and the net revenue formula must be switched as described in Section 2. If the fees look like realistic dollar amounts, the current calculation stands.

## 4. Wholesale net revenue by product line, month and warehouse

**Question (Board of Directors).** What is the net revenue of each product line for wholesale orders, and how does it vary by month and warehouse?

**Approach.** Filter to `Wholesale` orders, group by product line, month and warehouse, and compute net revenue. Output is ordered by product line, then chronologically, then by net revenue within each month.

In [29]:
-- Wholesale net revenue by product line, month and warehouse
SELECT
    product_line,
    TO_CHAR(date, 'FMMonth')                      AS month,
    warehouse,
    ROUND(SUM(total - payment_fee)::numeric, 2)   AS net_revenue
FROM sales
WHERE client_type = 'Wholesale'
GROUP BY product_line, EXTRACT(MONTH FROM date), TO_CHAR(date, 'FMMonth'), warehouse
ORDER BY product_line, EXTRACT(MONTH FROM date), net_revenue DESC;

,product_line,month,warehouse,net_revenue
0,Braking system,June,Central,3684.89
1,Braking system,June,North,1487.77
2,Braking system,June,West,1212.75
3,Braking system,July,Central,3778.65
4,Braking system,July,West,3060.93
5,Braking system,July,North,2594.44
6,Braking system,August,Central,3039.41
7,Braking system,August,West,2500.67
8,Braking system,August,North,1770.84
9,Electrical system,June,Central,2904.93


**Findings** (from the previous execution)

- **Largest single cells:** Engine at Central in August (USD 9,528.71), Frame & body at Central in August (USD 8,657.99), Suspension & traction at North in June (USD 8,065.74) and Frame & body at North in August (USD 7,898.89).
- **Engine is lumpy.** Central's wholesale Engine revenue moved from USD 6,548.85 (June) to USD 1,827.03 (July) and back to USD 9,528.71 (August), a pattern typical of a few large, irregular bulk orders. North has Engine sales only in July and August, and **West has none in any month**.
- **West contributes little in most lines.** Its largest wholesale cells are Suspension & traction (June–July) and Frame & body (June); its smallest is Electrical system in July (USD 449.46).
- **Planning implication.** Wholesale revenue depends on a handful of large orders, so single-month swings should not be read as trends.

## 5. Payment-method profitability by warehouse and month

**Question (Finance).** Which payment method generates the most net revenue for each warehouse in each month?

**Approach.** Compute net revenue per payment method for every warehouse-month, rank within each combination with `RANK()`, and keep the top method together with its share of that warehouse-month's net revenue.

In [30]:
-- Top payment method by net revenue for each warehouse and month
WITH payment_net AS (
    SELECT
        warehouse,
        EXTRACT(MONTH FROM date)::int                 AS month_num,
        TO_CHAR(date, 'FMMonth')                      AS month,
        payment,
        ROUND(SUM(total - payment_fee)::numeric, 2)   AS net_revenue
    FROM sales
    GROUP BY warehouse, 2, 3, payment
),
ranked AS (
    SELECT
        *,
        RANK() OVER (PARTITION BY warehouse, month_num ORDER BY net_revenue DESC) AS rnk,
        ROUND(100 * net_revenue
              / SUM(net_revenue) OVER (PARTITION BY warehouse, month_num), 1)     AS share_of_warehouse_month_pct
    FROM payment_net
)
SELECT warehouse, month, payment, net_revenue, share_of_warehouse_month_pct
FROM ranked
WHERE rnk = 1
ORDER BY warehouse, month_num;

,warehouse,month,payment,net_revenue,share_of_warehouse_month_pct
0,Central,June,Transfer,23453.08,53.2
1,Central,July,Transfer,23893.59,49.5
2,Central,August,Transfer,31509.00,63.6
3,North,June,Transfer,17000.12,51.0
4,North,July,Transfer,17585.25,60.4
5,North,August,Transfer,23480.13,62.2
6,West,June,Transfer,8645.98,48.4
7,West,July,Transfer,7606.51,47.1
8,West,August,Transfer,6466.42,50.1


**Findings** (from the previous execution)

- **Transfer ranks first in all nine warehouse-month combinations.**
- **Direction of travel differs by warehouse.** Transfer net revenue at Central rose 1.9% from June to July and 31.9% from July to August; at North, 3.4% and 33.5%. At West it fell 12.0% and then 15.0%, a cumulative decline of about 25% from June (USD 8,645.98) to August (USD 6,466.42).
- **Interpretation caution.** Being first by net revenue shows that Transfer carries the most volume, not that it is the cheapest method to process. The cost view below puts the fee burden of each method next to its revenue share.

In [31]:
-- Cost-efficiency of each payment method across the whole network
SELECT
    payment,
    COUNT(DISTINCT order_number)                                      AS orders,
    ROUND((100.0 * COUNT(DISTINCT order_number)
           / SUM(COUNT(DISTINCT order_number)) OVER ())::numeric, 1)  AS share_of_orders_pct,
    ROUND(SUM(total)::numeric, 2)                                     AS gross_revenue,
    ROUND(SUM(payment_fee)::numeric, 2)                               AS total_fees,
    ROUND(SUM(total - payment_fee)::numeric, 2)                       AS net_revenue,
    ROUND((100 * SUM(payment_fee) / SUM(total))::numeric, 2)          AS effective_fee_rate_pct,
    ROUND((SUM(total - payment_fee) / COUNT(DISTINCT order_number))::numeric, 2) AS net_revenue_per_order
FROM sales
GROUP BY payment
ORDER BY effective_fee_rate_pct DESC;

,payment,orders,share_of_orders_pct,gross_revenue,total_fees,net_revenue,effective_fee_rate_pct,net_revenue_per_order
0,Credit card,659,65.9,110271.57,19.77,110251.80,0.02,167.30
1,Cash,116,11.6,19199.10,0.00,19199.10,0.00,165.51
2,Transfer,225,22.5,159642.33,2.25,159640.08,0.00,709.51


**How Finance can use this.** Compare `effective_fee_rate_pct` across methods against each method's share of orders. A method with a high fee rate and a large order share is the main lever for reducing transaction costs (for example, by incentivizing a cheaper method on large wholesale orders). Interpret the absolute rates only after the Section 3 diagnostic confirms how `payment_fee` is stored.

## 6. Most-ordered product lines by client type

**Question (Marketing).** What are the three most ordered product lines for retail and for wholesale customers?

**Approach.** Rank product lines by total units ordered within each client type. `RANK()` is used so that tied lines are not dropped.

In [32]:
-- Top 3 product lines by units ordered, for each client type
WITH ranked_product_lines AS (
    SELECT
        client_type,
        product_line,
        SUM(quantity) AS total_quantity,
        RANK() OVER (PARTITION BY client_type ORDER BY SUM(quantity) DESC) AS rnk
    FROM sales
    GROUP BY client_type, product_line
)
SELECT client_type, product_line, total_quantity, rnk AS rank
FROM ranked_product_lines
WHERE rnk <= 3
ORDER BY client_type, rnk, product_line;

,client_type,product_line,total_quantity,rank
0,Retail,Suspension & traction,1017,1
1,Retail,Braking system,862,2
2,Retail,Electrical system,862,2
3,Wholesale,Braking system,1268,1
4,Wholesale,Suspension & traction,1128,2
5,Wholesale,Frame & body,932,3


**Findings** (from the previous execution)

| Client type | Rank | Product line | Units |
|---|---|---|---|
| Retail | 1 | Suspension & traction | 1,017 |
| Retail | 2 (tie) | Braking system | 862 |
| Retail | 2 (tie) | Electrical system | 862 |
| Wholesale | 1 | Braking system | 1,268 |
| Wholesale | 2 | Suspension & traction | 1,128 |
| Wholesale | 3 | Frame & body | 932 |

- **Suspension & traction and Braking system are top-two for both channels**, making them natural anchors for any network-wide campaign.
- **Channel-specific themes:** Electrical system for retail, Frame & body for wholesale.
- Braking system is the most ordered wholesale line but a modest revenue contributor (Section 7), so campaign planning should consider revenue per unit, not units alone.

## 7. Product-line Pareto (ABC) analysis

**Question.** Which product lines generate most of the net revenue, and how does that compare with the units they move?

**Approach.** Sort product lines by net revenue, compute each line's cumulative share of total net revenue, and classify them: **A** up to 70%, **B** up to 90%, **C** above 90%. Net revenue per unit is added to separate high-value from high-volume lines.

In [33]:
-- Cumulative Pareto (ABC) analysis of product lines by net revenue
WITH product_summary AS (
    SELECT
        product_line,
        SUM(quantity)                              AS total_units,
        ROUND(SUM(total - payment_fee)::numeric, 2) AS net_revenue
    FROM sales
    GROUP BY product_line
),
cumulative_calc AS (
    SELECT
        product_line,
        total_units,
        net_revenue,
        ROUND((100 * net_revenue / SUM(net_revenue) OVER ())::numeric, 2) AS revenue_share_pct,
        ROUND((100 * SUM(net_revenue) OVER (ORDER BY net_revenue DESC
                                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
               / SUM(net_revenue) OVER ())::numeric, 2)                    AS cum_revenue_pct,
        ROUND((net_revenue / total_units)::numeric, 2)                     AS net_revenue_per_unit
    FROM product_summary
)
SELECT
    product_line,
    total_units,
    net_revenue,
    net_revenue_per_unit,
    revenue_share_pct,
    cum_revenue_pct,
    CASE
        WHEN cum_revenue_pct <= 70 THEN 'A'
        WHEN cum_revenue_pct <= 90 THEN 'B'
        ELSE 'C'
    END AS abc_category
FROM cumulative_calc
ORDER BY net_revenue DESC;

,product_line,total_units,net_revenue,net_revenue_per_unit,revenue_share_pct,cum_revenue_pct,abc_category
0,Suspension & traction,2145,73009.11,34.04,25.25,25.25,A
1,Frame & body,1619,69021.08,42.63,23.88,49.13,A
2,Electrical system,1698,43608.40,25.68,15.08,64.21,A
3,Braking system,2130,38345.10,18.00,13.26,77.48,B
4,Engine,627,37943.99,60.52,13.13,90.60,C
5,Miscellaneous,1176,27163.30,23.10,9.40,100.00,C


**Findings** (from the previous execution; revenue per unit and revenue share are derived from the same figures)

| Product line | Units | Net revenue | Net revenue / unit | Revenue share | Cumulative | Class |
|---|---:|---:|---:|---:|---:|:--:|
| Suspension & traction | 2,145 | USD 73,009.11 | USD 34.04 | 25.3% | 25.25% | A |
| Frame & body | 1,619 | USD 69,021.08 | USD 42.63 | 23.9% | 49.13% | A |
| Electrical system | 1,698 | USD 43,608.40 | USD 25.68 | 15.1% | 64.21% | A |
| Braking system | 2,130 | USD 38,345.10 | USD 18.00 | 13.3% | 77.48% | B |
| Engine | 627 | USD 37,943.99 | USD 60.52 | 13.1% | 90.60% | C |
| Miscellaneous | 1,176 | USD 27,163.30 | USD 23.10 | 9.4% | 100.00% | C |

- **Three "A" lines produce 64% of revenue** and should receive priority for stock availability and service levels.
- **Braking system is high-volume, low-value.** It is the second most ordered line yet earns about USD 18 per unit, against a network average of about USD 30.77. Review pricing, bundling and margin before expanding promotions.
- **Engine is low-volume, high-value.** Per unit it earns more than three times the network average. It is classed C only because of its small unit count, and it sits just above the 90% boundary (Braking system leads it by about USD 401), so stock-outs here cost more per lost sale than in any other line.
- The ABC class depends on the chosen thresholds; lines close to a boundary (Braking system, Engine) should be treated as borderline.

## 8. Warehouse throughput and network load

**Question (Operations).** How is work distributed across the three warehouses, and where is congestion most likely?

**Scope note.** The dataset contains orders, units and dates but no capacity, staffing or fulfillment-time data. Volume share and peak-day load are therefore used as *proxies* for utilization; they cannot measure utilization directly.

In [34]:
-- Throughput and volume share by warehouse
SELECT
    warehouse,
    COUNT(DISTINCT order_number)                                        AS total_orders_handled,
    SUM(quantity)                                                       AS total_units_moved,
    ROUND(AVG(quantity)::numeric, 2)                                    AS avg_units_per_order,
    ROUND(SUM(total - payment_fee)::numeric, 2)                         AS net_throughput_value,
    ROUND((100.0 * SUM(quantity) / SUM(SUM(quantity)) OVER ())::numeric, 2) AS volume_share_pct,
    ROUND((100.0 * SUM(total - payment_fee)
           / SUM(SUM(total - payment_fee)) OVER ())::numeric, 2)        AS value_share_pct
FROM sales
GROUP BY warehouse
ORDER BY total_units_moved DESC;

,warehouse,total_orders_handled,total_units_moved,avg_units_per_order,net_throughput_value,volume_share_pct,value_share_pct
0,Central,480,4527,9.43,141972.13,48.19,49.11
1,North,340,3254,9.57,100196.26,34.64,34.66
2,West,180,1614,8.97,46922.59,17.18,16.23


**Findings** (from the previous execution)

| Warehouse | Orders | Units | Avg units / order | Net throughput value | Volume share |
|---|---:|---:|---:|---:|---:|
| Central | 480 | 4,527 | 9.43 | USD 141,972.13 | 48.19% |
| North | 340 | 3,254 | 9.57 | USD 100,196.26 | 34.64% |
| West | 180 | 1,614 | 8.97 | USD 46,922.59 | 17.18% |

- Central handles almost half of all units and a similar share of value, so any disruption or congestion there affects roughly half of the business.
- Average units per order are similar across sites (8.97–9.57), which indicates comparable order profiles; differences in load come from order *count*, not order size.

The next query tests the congestion hypothesis at day level: how many orders and units each warehouse processes per day, by month, and how high the busiest day gets.

In [35]:
-- Peak-day load by warehouse and month (proxy for congestion risk)
WITH daily AS (
    SELECT
        warehouse,
        date,
        COUNT(DISTINCT order_number) AS orders,
        SUM(quantity)                AS units
    FROM sales
    GROUP BY warehouse, date
)
SELECT
    warehouse,
    TO_CHAR(date, 'FMMonth')                 AS month,
    COUNT(*)                                 AS active_days,
    ROUND(AVG(orders)::numeric, 1)           AS avg_daily_orders,
    MAX(orders)                              AS peak_daily_orders,
    ROUND(AVG(units)::numeric, 0)            AS avg_daily_units,
    MAX(units)                               AS peak_daily_units,
    ROUND((MAX(units) / AVG(units))::numeric, 2) AS peak_to_average_ratio
FROM daily
GROUP BY warehouse, EXTRACT(MONTH FROM date), TO_CHAR(date, 'FMMonth')
ORDER BY warehouse, EXTRACT(MONTH FROM date);

,warehouse,month,active_days,avg_daily_orders,peak_daily_orders,avg_daily_units,peak_daily_units,peak_to_average_ratio
0,Central,June,30,5.2,11,47,128,2.72
1,Central,July,31,5.8,14,53,125,2.34
2,Central,August,28,5.1,13,52,189,3.62
3,North,June,30,3.9,8,35,79,2.24
4,North,July,31,3.3,7,31,83,2.64
5,North,August,28,4.3,9,44,132,3.02
6,West,June,27,2.4,7,21,102,4.78
7,West,July,31,2.0,6,17,57,3.33
8,West,August,24,2.2,5,21,82,3.88


**How to read it.** A warehouse with a high peak-to-average ratio, especially in August, has load that is hard to staff for. Confirming a true bottleneck also requires packing and staging capacity, shift data and dock-to-dispatch times, none of which are in this dataset.

## 9. Order profiling: retail vs wholesale

**Question.** How do order size and value differ by client type and warehouse?

In [36]:
-- Order profile by client type and warehouse
SELECT
    client_type,
    warehouse,
    COUNT(DISTINCT order_number)             AS order_count,
    SUM(quantity)                            AS total_units,
    ROUND(AVG(quantity)::numeric, 2)         AS avg_units_per_order,
    ROUND(AVG(unit_price)::numeric, 2)       AS avg_unit_price,
    ROUND(AVG(total)::numeric, 2)            AS avg_order_value
FROM sales
GROUP BY client_type, warehouse
ORDER BY client_type, warehouse;

,client_type,warehouse,order_count,total_units,avg_units_per_order,avg_unit_price,avg_order_value
0,Retail,Central,371,2039,5.50,30.28,170.15
1,Retail,North,263,1394,5.30,30.24,160.22
2,Retail,West,141,782,5.55,30.38,171.68
3,Wholesale,Central,109,2488,22.83,31.31,723.46
4,Wholesale,North,77,1860,24.16,30.72,754.11
5,Wholesale,West,39,832,21.33,27.49,582.55


**Findings** (from the previous execution)

| Client type | Warehouse | Orders | Units | Avg units / order | Avg unit price | Avg order value |
|---|---|---:|---:|---:|---:|---:|
| Retail | Central | 371 | 2,039 | 5.50 | USD 30.28 | USD 170.15 |
| Retail | North | 263 | 1,394 | 5.30 | USD 30.24 | USD 160.22 |
| Retail | West | 141 | 782 | 5.55 | USD 30.38 | USD 171.68 |
| Wholesale | Central | 109 | 2,488 | 22.83 | USD 31.31 | USD 723.46 |
| Wholesale | North | 77 | 1,860 | 24.16 | USD 30.72 | USD 754.11 |
| Wholesale | West | 39 | 832 | 21.33 | USD 27.49 | USD 582.55 |

- **Wholesale orders are about four times larger in units** and four to five times larger in value, and they account for 55.1% of all units from 22.5% of orders. This makes wholesale operations sensitive to a small number of accounts.
- **Retail is uniform across warehouses** (about USD 30 per unit, USD 160–USD 172 per order), so differences between sites are driven by wholesale.
- **West wholesale is priced and sized lower.** Its average unit price (USD 27.49) is about 12% below Central (USD 31.31) and 10% below North (USD 30.72), and its average order value is the lowest of all wholesale segments. A product-mix effect is a plausible contributor: with no Engine sales (the highest-value line), the West basket is weighted toward lower-priced lines. Discounting cannot be ruled out with this data.

## 10. Investigation: the West warehouse Engine gap

**Observation.** Wholesale Engine sales are absent at West in every month (Section 4), and absent at North in June. West's sales are also lower overall (Sections 5 and 8).

**Competing explanations**

1. **Supply:** Engine parts were out of stock at West, for example through allocation from the main distribution center.
2. **Demand:** the West region has few B2B customers that repair or overhaul engines.

The two queries below use the available data to discriminate between these explanations.

In [37]:
-- Engine activity by warehouse, client type and month
SELECT
    warehouse,
    client_type,
    TO_CHAR(date, 'FMMonth')                 AS month,
    COUNT(DISTINCT order_number)             AS orders,
    SUM(quantity)                            AS units,
    ROUND(SUM(total)::numeric, 2)            AS gross_revenue
FROM sales
WHERE product_line = 'Engine'
GROUP BY warehouse, client_type, EXTRACT(MONTH FROM date), TO_CHAR(date, 'FMMonth')
ORDER BY warehouse, client_type, EXTRACT(MONTH FROM date);

,warehouse,client_type,month,orders,units,gross_revenue
0,Central,Retail,June,10,45,2707.12
1,Central,Retail,July,9,61,3653.78
2,Central,Retail,August,6,51,2997.54
3,Central,Wholesale,June,4,104,6548.89
4,Central,Wholesale,July,1,32,1827.04
5,Central,Wholesale,August,6,156,9528.77
6,North,Retail,June,4,19,1158.45
7,North,Retail,July,5,29,1698.46
8,North,Retail,August,5,18,1048.47
9,North,Wholesale,July,1,16,1007.15


In [38]:
-- Engine as a share of each warehouse's units, split by client type
SELECT
    warehouse,
    client_type,
    COALESCE(SUM(quantity) FILTER (WHERE product_line = 'Engine'), 0) AS engine_units,
    SUM(quantity)                                                     AS all_units,
    ROUND((100.0 * COALESCE(SUM(quantity) FILTER (WHERE product_line = 'Engine'), 0)
           / SUM(quantity))::numeric, 1)                              AS engine_share_pct
FROM sales
GROUP BY warehouse, client_type
ORDER BY warehouse, client_type;

,warehouse,client_type,engine_units,all_units,engine_share_pct
0,Central,Retail,157,2039,7.7
1,Central,Wholesale,292,2488,11.7
2,North,Retail,66,1394,4.7
3,North,Wholesale,56,1860,3.0
4,West,Retail,56,782,7.2
5,West,Wholesale,0,832,0.0


**Decision guide**

| What the data shows | Most likely explanation | Suggested action |
|---|---|---|
| West has retail Engine sales but no wholesale Engine sales | Stock is available; the gap is on the demand side (no B2B engine-repair customers) | Targeted B2B outreach in the West region, or accept the gap |
| West has no Engine sales at all, retail or wholesale | Supply or allocation problem | Check stock and allocation records; review the distribution plan |
| West had Engine sales early in the period that then stopped | Stock-out is more likely | Review replenishment history and lead times |
| Engine share at West is low but nonzero for both channels | Smaller, regional demand | Keep a small safety stock; do not over-allocate |

**Evidence outside this dataset** is needed to settle the question: warehouse stock and allocation records, customer counts by region, and lost-sale or back-order logs.

## 11. Recommendations and next steps

| # | Recommendation | Owner | Basis |
|---|---|---|---|
| 1 | **Confirm the `payment_fee` definition** and re-run the notebook; then circulate revenue and fee figures. | Data / Finance | Sections 2, 3 |
| 2 | **Review payment-method costs.** Compare effective fee rate against order share and consider incentives toward lower-cost methods on large wholesale orders. | Finance | Section 5 |
| 3 | **Protect availability of the A-class lines** (Suspension & traction, Frame & body, Electrical system), which generate about 64% of net revenue. | Operations | Section 7 |
| 4 | **Review pricing and bundling for Braking system.** High volume, about USD 18 per unit. | Product / Finance | Sections 6, 7 |
| 5 | **Monitor Engine stock levels closely.** Low volume but the highest revenue per unit, and wholesale demand is lumpy. | Operations | Sections 4, 7 |
| 6 | **Anchor campaigns on Suspension & traction and Braking system** (top two in both channels); add Electrical system for retail and Frame & body for wholesale. | Marketing | Section 6 |
| 7 | **Plan Central warehouse capacity for peak days,** and collect capacity and fulfillment-time data to confirm any bottleneck. | Operations | Section 8 |
| 8 | **Resolve the West Engine gap** using the Section 10 queries plus stock and customer records, and investigate West's falling Transfer revenue and lower wholesale pricing. | Operations / Sales | Sections 5, 9, 10 |

## 12. Limitations

- **Short window:** three months of data limits trend and seasonality conclusions; the August increase at Central and North may be seasonal or a one-off.
- **Lumpy wholesale demand:** a few large orders can move monthly totals, so single-month changes should be read cautiously.
- **No cost data:** results are revenue-based; product-line profitability would require cost of goods sold.
- **No capacity, inventory or customer identifiers:** bottleneck, stock-out and customer-concentration questions can only be tested indirectly.
- **Payment-fee definition** is unresolved until Section 3 is reviewed.
- **Narrative figures** in the findings blocks come from the previous run of the original queries; re-run the notebook to refresh all tables, then check the narrative against the new outputs.